In [13]:
# build_matrices.ipynb — full code for Cells 1 and 2

# ------------------------------
# Cell 1: Imports and Config
# ------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel, wilcoxon
from sklearn.metrics import cohen_kappa_score

# Set root path and file locations
project_root = Path("..")
input_dir = project_root / "postproc_intermediate"
output_base = input_dir / "matrices"

# Input tidyLong files (produced by build_tidyLong.ipynb)
tidy_files = {
    "llama": input_dir / "llama_tidyLong.csv",
    "gpt": input_dir / "gpt_tidyLong.csv"
}

In [14]:
# ------------------------------
# Cell 2: Load tidyLong Files
# ------------------------------
all_dfs = []

for model, path in tidy_files.items():
    if not path.exists():
        print(f"⚠ Skipping missing file: {path.name}")
        continue
    df = pd.read_csv(path)
    df["model"] = model
    all_dfs.append(df)

if not all_dfs:
    raise RuntimeError("No tidyLong input files were found.")

# Combine into one tidy dataframe
df_all = pd.concat(all_dfs, ignore_index=True)

# Quick peek
print(f"Loaded {df_all.shape[0]} rows from tidyLong files")
print("Columns:", df_all.columns.tolist())
print("Models:", df_all.model.unique())
print("References:", df_all.reference.unique())


Loaded 106094 rows from tidyLong files
Columns: ['patient_id', 'pred_score', 'extract', 'prompt', 'adversarial', 'model', 'gold_NIHSS_total', 'gold_1a', 'gold_1b', 'gold_1c', 'gold_2', 'gold_3', 'gold_4', 'gold_5a', 'gold_5b', 'gold_6a', 'gold_6b', 'gold_7', 'gold_8', 'gold_9', 'gold_10', 'gold_11', 'silver_NIHSS_total', 'silver_1a', 'silver_1b', 'silver_1c', 'silver_2', 'silver_3', 'silver_4', 'silver_5a', 'silver_5b', 'silver_6a', 'silver_6b', 'silver_7', 'silver_8', 'silver_9', 'silver_10', 'silver_11', 'admit_date', 'year_bin', 'source_silver', 'dataset_source_flag', 'fold', 'reference', 'abs_error', 'squared_error']
Models: ['llama' 'gpt']
References: ['gold' 'silver']


In [15]:
# ------------------------------
# Cell 3: Define Metric Functions
# ------------------------------
def compute_qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def compute_pearson(y_true, y_pred):
    return np.corrcoef(y_true, y_pred)[0, 1]

def compute_mae(abs_errors):
    return abs_errors.mean()

def compute_rmse(squared_errors):
    return np.sqrt(squared_errors.mean())

def compute_paired_ttest(y_true, y_pred):
    stat, pval = ttest_rel(y_pred, y_true)
    mean_diff = (y_pred - y_true).mean()
    return {
        "mean_diff": mean_diff,
        "t_stat": stat,
        "pValue": pval
    }

def compute_wilcoxon(y_true, y_pred):
    try:
        stat, pval = wilcoxon(y_pred, y_true)
    except ValueError:
        stat, pval = np.nan, np.nan  # e.g., constant values
    return {
        "wilcoxon_stat": stat,
        "wilcoxon_p": pval
    }

def icc_a1_from_two_raters(y1, y2):
    """
    Two-way mixed, absolute agreement, single measure ICC (A,1).
    y1, y2: arrays of shape (n,) for the two raters (e.g., model vs reference)
    Returns float in [-1, 1].
    """
    y1 = np.asarray(y1, dtype=float)
    y2 = np.asarray(y2, dtype=float)
    mask = ~np.isnan(y1) & ~np.isnan(y2)
    y1, y2 = y1[mask], y2[mask]
    n = len(y1)
    if n < 2:
        return np.nan

    # Arrange n×k data (k=2 raters: columns = [rater1, rater2])
    Y = np.vstack([y1, y2]).T
    n, k = Y.shape  # k = 2

    # Grand mean
    gm = Y.mean()

    # Subject means and Rater means
    subj_means = Y.mean(axis=1, keepdims=True)
    rater_means = Y.mean(axis=0, keepdims=True)

    # Sum of squares
    ss_total = ((Y - gm)**2).sum()
    ss_rows  = (k * (subj_means.squeeze() - gm)**2).sum()           # subjects
    ss_cols  = (n * (rater_means.squeeze() - gm)**2).sum()           # raters
    ss_err   = ss_total - ss_rows - ss_cols

    # Mean squares
    ms_rows = ss_rows / (n - 1) if n > 1 else np.nan       # MSR
    ms_cols = ss_cols / (k - 1) if k > 1 else np.nan       # MSC
    ms_err  = ss_err  / ((n - 1) * (k - 1)) if n > 1 else np.nan  # MSE

    # ICC(A,1) (McGraw & Wong): (MSR - MSE) / (MSR + (k-1)MSE + k*(MSC - MSE)/n)
    denom = ms_rows + (k - 1) * ms_err + (k * (ms_cols - ms_err) / n)
    if denom == 0:
        return np.nan
    return (ms_rows - ms_err) / denom



In [16]:
# ------------------------------
# Cell 4: Define Subgrouping Logic
# ------------------------------
def get_subgroup_slices(df):
    """
    Yield (subgrp_type, subgrp_val, df_subset) tuples.

    Rules:
    - "overall", "inst", and "time" subgroups are computed only on NON-adversarial rows.
    - "adv" subgroup is computed only on adversarial rows (adversarial == True).
    """

    # Non-adversarial subset
    if "adversarial" in df.columns:
        non_adv = df[df["adversarial"] == False]
    else:
        non_adv = df.copy()

    # Overall (non-adversarial only)
    if not non_adv.empty:
        yield ("overall", "all", non_adv)

    # Institution subgroups (non-adversarial only)
    source_map = {0: "BMC", 1: "MGB", 2: "MIMIC"}
    if "dataset_source_flag" in non_adv.columns:
        for code, name in source_map.items():
            subdf = non_adv[non_adv["dataset_source_flag"] == code]
            if not subdf.empty:
                yield ("inst", name, subdf)

    # Timing subgroups (non-adversarial only)
    if "year_bin" in non_adv.columns:
        for val in non_adv["year_bin"].dropna().unique():
            subdf = non_adv[non_adv["year_bin"] == val]
            if not subdf.empty:
                yield ("time", val, subdf)

    # Adversarial subgroup (adversarial only)
    if "adversarial" in df.columns:
        adv = df[df["adversarial"] == True]
        if not adv.empty:
            yield ("adv", "abbrReintro", adv)


In [17]:
#####
## Cell 4a
#####

from collections import defaultdict

# One-shot cohort-wide stats
stats_per_subgroup = defaultdict(dict)

for model in df_all["model"].unique():
    df_model = df_all[df_all["model"] == model]

    for reference in ["gold", "silver"]:
        df_ref = df_model[df_model["reference"] == reference]

        for (subgrp_type, subgrp_val, df_sub) in get_subgroup_slices(df_ref):
            for (extract, df_ext) in df_sub.groupby("extract"):
                for (prompt, df_prompt) in df_ext.groupby("prompt"):

                    y_true = df_prompt[f"{reference}_NIHSS_total"].astype(int)
                    y_pred = df_prompt["pred_score"].astype(int)

                    ttest = compute_paired_ttest(y_true, y_pred)
                    wilx = compute_wilcoxon(y_true, y_pred)

                    key = (model, reference, prompt, extract, subgrp_type, subgrp_val)
                    stats_per_subgroup[key].update(ttest)
                    stats_per_subgroup[key].update(wilx)



In [18]:
#####
## Cell 4b
#####

def format_statistical_string(mean_diff, t_stat, pval_t, w_stat, pval_w):
    """Return LaTeX-ready delta/t and Wilcoxon strings based on p-value thresholds."""
    def star(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    delta_t_str = f"${mean_diff:+.2f}$, $t{{={t_stat:.1f}}}$ {star(pval_t)}"
    w_str = f"$W{{={int(round(w_stat)):,}}}$ {star(pval_w)}"
    return delta_t_str.strip(), w_str.strip()



In [19]:
#####
## Cell 4c
#####

# Convert stats to formatted LaTeX strings
formatted_stats = defaultdict(lambda: defaultdict(dict))

for key, stats in stats_per_subgroup.items():
    model, reference, prompt, extract, subgrp_type, subgrp_val = key  # unpack in model-first order

    if not all(pd.notnull([stats["mean_diff"], stats["t_stat"], stats["pValue"],
                           stats["wilcoxon_stat"], stats["wilcoxon_p"]])):
        continue

    delta_str, w_str = format_statistical_string(
        mean_diff=stats["mean_diff"],
        t_stat=stats["t_stat"],
        pval_t=stats["pValue"],
        w_stat=stats["wilcoxon_stat"],
        pval_w=stats["wilcoxon_p"]
    )

    formatted_stats[(model, reference, "delta_t", subgrp_type, subgrp_val)][(extract, prompt)] = delta_str
    formatted_stats[(model, reference, "w", subgrp_type, subgrp_val)][(extract, prompt)] = w_str


In [20]:
# ------------------------------
# Cell 5: Compute Metrics Per Fold × Subgroup
# ------------------------------
from collections import defaultdict

# Structure: {(model, reference, prompt, extract, subgrp_type, subgrp_val, fold): metrics}
per_fold_metrics = defaultdict(lambda: defaultdict(dict))

folds = sorted(df_all["fold"].dropna().unique())

for model in df_all["model"].unique():
    df_model = df_all[df_all["model"] == model]

    for reference in ["gold", "silver"]:
        df_ref = df_model[df_model["reference"] == reference]

        for fold in folds:
            df_fold = df_ref[df_ref["fold"] == fold]

            for (subgrp_type, subgrp_val, df_sub) in get_subgroup_slices(df_fold):
                for (extract, df_ext) in df_sub.groupby("extract"):
                    for (prompt, df_prompt) in df_ext.groupby("prompt"):

                        y_true = df_prompt[f"{reference}_NIHSS_total"].astype(int)
                        y_pred = df_prompt["pred_score"].astype(int)
                        abs_error = df_prompt["abs_error"]
                        squared_error = df_prompt["squared_error"]

                        # Note: (model, reference, …) to mirror directory structure
                        row_key = (model, reference, prompt, extract, subgrp_type, subgrp_val, fold)

                        per_fold_metrics[row_key]["qwk"] = compute_qwk(y_true, y_pred)
                        per_fold_metrics[row_key]["pearson"] = compute_pearson(y_true, y_pred)
                        per_fold_metrics[row_key]["icc3"] = icc_a1_from_two_raters(y_true, y_pred)
                        per_fold_metrics[row_key]["mae"] = compute_mae(abs_error)
                        per_fold_metrics[row_key]["rmse"] = compute_rmse(squared_error)

                        # Optionally: add paired t-tests / Wilcoxon stats
                        # ttest = compute_paired_ttest(y_true, y_pred)
                        # wilx = compute_wilcoxon(y_true, y_pred)
                        # per_fold_metrics[row_key].update(ttest)
                        # per_fold_metrics[row_key].update(wilx)


In [25]:
import numpy as np

# assuming per_fold_metrics is already populated
tolerance = 1e-12
matches = []
diffs = []

for key, metrics in per_fold_metrics.items():
    icc = metrics.get("icc3", np.nan)
    qwk = metrics.get("qwk", np.nan)
    if np.isnan(icc) or np.isnan(qwk):
        continue
    diff = abs(icc - qwk)
    diffs.append(diff)
    if diff < tolerance:
        matches.append(key)

print(f"Total comparisons: {len(diffs)}")
print(f"Exact/near-identical (|ICC-QWK| < {tolerance}): {len(matches)}")
print(f"Mean |ICC-QWK| difference: {np.nanmean(diffs):.6g}")
print(f"Max |ICC-QWK| difference:  {np.nanmax(diffs):.6g}")

# Optional: print a few matching keys for inspection
if matches:
    print("\nExamples of identical entries:")
    for k in matches[:5]:
        print(k, per_fold_metrics[k])


Total comparisons: 1320
Exact/near-identical (|ICC-QWK| < 1e-12): 0
Mean |ICC-QWK| difference: 0.00246603
Max |ICC-QWK| difference:  0.0287703


In [21]:
# ------------------------------
# Cell 6: Aggregate Across Folds (model → reference order)
# ------------------------------
from collections import defaultdict
from itertools import groupby

# Aggregate containers
aggregate_metrics = defaultdict(lambda: defaultdict(dict))  
# Key: (model, reference, metric, subgrp_type, subgrp_val)
# Value: {kind → {(extract, prompt): value}}

def get_group_key(key):
    return key[:-1]  # all but fold

sorted_keys = sorted(per_fold_metrics.keys(), key=get_group_key)

for group_key, group in groupby(sorted_keys, key=get_group_key):
    metrics_by_fold = [per_fold_metrics[k] for k in group]

    # Unpack in (model, reference, …) order
    model, reference, prompt, extract, subgrp_type, subgrp_val = group_key

    for metric in metrics_by_fold[0].keys():
        if metric in {"pValue", "t_stat", "mean_diff", "wilcoxon_stat", "wilcoxon_p"}:
            continue

        values = [m[metric] for m in metrics_by_fold if pd.notnull(m[metric])]
        if not values:
            continue

        mean = np.mean(values)
        std = np.std(values)
        pretty = f"{mean:.2f} ± {std:.2f}"

        # Keep model outside reference in the key
        matrix_key = (model, reference, metric, subgrp_type, subgrp_val)
        cell_key = (extract, prompt)

        aggregate_metrics[matrix_key]["mean"][cell_key] = mean
        aggregate_metrics[matrix_key]["stdDev"][cell_key] = std
        aggregate_metrics[matrix_key]["pretty"][cell_key] = pretty


In [22]:
# ------------------------------
# Cell 7: Write Matrix Files (model → reference order)
# ------------------------------
def write_matrix(df_dict, model, reference, metric, subgrp_type, subgrp_val):
    for kind, matrix_dict in df_dict.items():
        df = pd.DataFrame.from_dict(matrix_dict, orient="index")
        df.index = pd.MultiIndex.from_tuples(df.index, names=["extract", "prompt"])
        df = df.unstack(level="prompt")
        df.columns = df.columns.droplevel(0)

        outdir = project_root / "postproc_intermediate" / "matrices" / model / reference / "aggregate"
        outdir.mkdir(parents=True, exist_ok=True)

        fname = f"{metric}_{subgrp_type}_{subgrp_val}_{kind}.csv"
        path = outdir / fname
        df.to_csv(path)
        # print(f"✔ Saved {kind}: {path.relative_to(project_root)} ({df.shape})")

# Iterate over (model, reference, …) keys
for (model, reference, metric, subgrp_type, subgrp_val), df_dict in aggregate_metrics.items():
    write_matrix(df_dict, model, reference, metric, subgrp_type, subgrp_val)


In [23]:
# ------------------------------
# Cell 7a: WRITE deltaT and w FILES with camelCase NAMES (model → reference order)
# ------------------------------
for (model, reference, metric, subgrp_type, subgrp_val), matrix_dict in formatted_stats.items():
    # Apply naming convention: camelCase metric field
    if metric == "delta_t":
        metric = "deltaT"

    df = pd.DataFrame.from_dict(matrix_dict, orient="index")
    df.index = pd.MultiIndex.from_tuples(df.index, names=["extract", "prompt"])
    df = df.unstack(level="prompt")
    df.columns = df.columns.droplevel(0)

    outdir = project_root / "postproc_intermediate" / "matrices" / model / reference / "aggregate"
    outdir.mkdir(parents=True, exist_ok=True)

    fname = f"{metric}_{subgrp_type}_{subgrp_val}_pretty.csv"
    path = outdir / fname
    df.to_csv(path)
    # print(f"✔ Saved cohort-wide pretty: {path.relative_to(project_root)} ({df.shape})")


In [24]:
# ------------------------------
# Cell 8: Write Per-Fold Tidy Matrices (model → reference order)
# ------------------------------
per_fold_matrix_data = defaultdict(lambda: defaultdict(dict))

for key, metrics in per_fold_metrics.items():
    model, reference, prompt, extract, subgrp_type, subgrp_val, fold = key

    for metric, value in metrics.items():
        if pd.isnull(value):
            continue
        matrix_key = (model, reference, metric, subgrp_type, subgrp_val, fold)
        cell_key = (extract, prompt)
        per_fold_matrix_data[matrix_key][cell_key] = value

for (model, reference, metric, subgrp_type, subgrp_val, fold), matrix_dict in per_fold_matrix_data.items():
    df = pd.DataFrame.from_dict(matrix_dict, orient="index")
    df.index = pd.MultiIndex.from_tuples(df.index, names=["extract", "prompt"])
    df = df.unstack(level="prompt")
    df.columns = df.columns.droplevel(0)

    outdir = project_root / "postproc_intermediate" / "matrices" / model / reference / "per_fold"
    outdir.mkdir(parents=True, exist_ok=True)

    fname = f"{metric}_{subgrp_type}_{subgrp_val}_fold{fold}.csv"
    path = outdir / fname
    df.to_csv(path)
    print(f"✔ Saved per-fold: {path.relative_to(project_root)} ({df.shape})")


✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\qwk_overall_all_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\pearson_overall_all_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\icc3_overall_all_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\mae_overall_all_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\rmse_overall_all_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\qwk_inst_BMC_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\pearson_inst_BMC_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\icc3_inst_BMC_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrices\llama\gold\per_fold\mae_inst_BMC_fold1.0.csv ((3, 6))
✔ Saved per-fold: postproc_intermediate\matrice